# 🔬 GIADA Task 2b — Identificabilità di `h_inf` e `tau_h`
Matrice 2×2 appaiata: rate supervision × loss multi-orizzonte, con direct-z Task 2 congelato come controllo positivo.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
WORKSPACE=Path('/kaggle/working/giada_task_2b'); GIADA_REPO=WORKSPACE/'giada'; TEACHER_REPO=WORKSPACE/'neuron_as_deep_net'
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_gate_h_rate_identifiability')
def run(args,cwd=None): print('+',' '.join(map(str,args))); subprocess.run(list(map(str,args)),cwd=cwd,check=True)


In [ ]:
WORKSPACE.mkdir(parents=True,exist_ok=True)
if not GIADA_REPO.exists(): run(['git','clone','--branch','codex/surrogate-validity-audit','--single-branch','https://github.com/Zagred47/giada.git',GIADA_REPO])
else: run(['git','fetch','origin','codex/surrogate-validity-audit'],cwd=GIADA_REPO); run(['git','checkout','--detach','FETCH_HEAD'],cwd=GIADA_REPO)
if not TEACHER_REPO.exists(): run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',TEACHER_REPO])
run(['git','checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],cwd=TEACHER_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=GIADA_REPO,text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 2b richiede GPU CUDA.'
from src.giada_teacher import (ExtractedGateFormula,GateHIdentifiabilityConfig,prepare_gate_h_identifiability,run_gate_h_identifiability,evaluate_gate_h_identifiability,verified_task2_artifact_root)
prereg=json.loads((GIADA_REPO/'experiments/teacher_gate_h_identifiability_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'task':prereg['name'],'factorial':prereg['factorial']})


In [ ]:
INPUT_ROOT=Path('/kaggle/input'); candidates=list(INPUT_ROOT.rglob('giada_gate_h_atomic_playground.zip'))
for p in INPUT_ROOT.rglob('final_report.json'):
    try:
        if json.loads(p.read_text()).get('schema_version')=='giada-task2-final-v1': candidates.append(p.parent)
    except Exception: pass
TASK2_SOURCE=next((p for p in candidates if p.exists()),None)
assert TASK2_SOURCE is not None,'Artefatto giada_gate_h_atomic_playground non trovato negli Input Kaggle.'
TASK2_ROOT=verified_task2_artifact_root(TASK2_SOURCE,'/kaggle/working/.giada_task2b_task2_cache')
print({'task2_source':str(TASK2_SOURCE),'verified_root':str(TASK2_ROOT)})


In [ ]:
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_gate_h_identifiability(formula); config=GateHIdentifiabilityConfig()
display({'contract':bundle['contract'],'objectives':config.objectives,'checkpoints':config.checkpoints})


In [ ]:
training_report=run_gate_h_identifiability(bundle,OUTPUT_DIR,config)
display({'valid':training_report['valid'],'winner':training_report['winner'],'selection':training_report['selection']})


In [ ]:
final_report=evaluate_gate_h_identifiability(bundle,OUTPUT_DIR,TASK2_ROOT,config)
display({'valid':final_report['valid'],'decision':final_report['decision'],'fresh_not_used_for_selection':not final_report['selection_used_fresh']})
assert final_report['valid'] and not final_report['selection_used_fresh']


## 📦 Download robusto
Download Blob/base64 già validato per Kaggle.

In [ ]:
from base64 import b64encode
from IPython.display import Javascript,display
archive=Path(shutil.make_archive('/kaggle/working/giada_gate_h_rate_identifiability','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name))
payload=b64encode(archive.read_bytes()).decode('ascii'); filename=archive.name
display(Javascript(f"""const raw=atob('{payload}');const bytes=new Uint8Array(raw.length);for(let i=0;i<raw.length;i++)bytes[i]=raw.charCodeAt(i);const url=URL.createObjectURL(new Blob([bytes],{{type:'application/zip'}}));const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})
